In [40]:
from __future__ import annotations

from pathlib import Path

import torch

from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.candidates.random import (
    RandomSemanticCandidateConfig,
    RandomSemanticIDCandidateSampler,
)
from flowrec.rewards.amazon_log import AmazonLogRewardConfig, AmazonLogRewardModel
from flowrec.ft_methods.gflow.config import GFlowConfig
from flowrec.ft_methods.gflow.method import GFlowMethod
from flowrec.training.trainer import Trainer, TrainerConfig

# Import your TIGER model from the repo.
# Adjust this import/constructor to the exact model API in your codebase.
from flowrec.models.tiger import TigerModel


AMAZON_DATASETS = {
    "beauty": {
        "reviews_path": "../data/raw/reviews_Beauty.json.gz",
        "metadata_path": "../data/raw/meta_Beauty.json.gz",
        "item_key": "asin",
        # Must be keyed by ASIN because item_key="asin".
        "semantic_id_path": "../data/processed/semantic_ids/beauty_rqvae_asin.json",
    },
    "instruments": {
        "reviews_path": "../data/raw/Musical_Instruments.json.gz",
        "metadata_path": "../data/raw/meta_Musical_Instruments.json.gz",
        "item_key": "asin",
        # Must be keyed by ASIN because item_key="asin".
        "semantic_id_path": "../artifacts/tokenizers/rqvae_musical_instruments/semantic_ids.json",
    },
}


In [41]:
def build_amazon_datamodule(
    dataset_name: str,
    *,
    seed: int = 42,
) -> AmazonReviewDataModule:
    dataset_cfg = AMAZON_DATASETS[dataset_name]

    cfg = AmazonReviewDataConfig(
        reviews_path=dataset_cfg["reviews_path"],
        metadata_path=dataset_cfg["metadata_path"],
        semantic_id_path=dataset_cfg["semantic_id_path"],
        item_key=dataset_cfg.get("item_key", "asin"),

        # Split.
        split_strategy="leave_one_out",
        global_eval_target_mode="last_per_user",
        global_history_policy="rolling",

        # Train example construction.
        train_all_prefixes=True,

        # Filtering.
        min_user_interactions=5,
        min_item_interactions=5,
        max_history_len=20,
        deduplicate_consecutive=True,

        # Important for train-only catalog after split.
        filter_cold_items_after_split=True,
        cold_item_train_source="observed",

        # Batching.
        batch_size=256,
        eval_batch_size=256,
        num_workers=0,
        pin_memory=True,
        persistent_workers=False,

        # Semantic IDs.
        offset_semantic_ids_by_one=True,
        drop_items_missing_semantic_id=True,

        seed=seed,
    )

    dm = AmazonReviewDataModule(cfg)
    dm.setup()

    return dm


def build_semantic_tables(
    dm: AmazonReviewDataModule,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Build mappings needed by random sampler and AmazonLogRewardModel.

    item_semantic_ids:
        semantic table, shape [num_semantic_rows, sid_len]

    item_id_by_semantic_row:
        semantic row -> internal atomic item ID

    item_id_to_semantic_row:
        internal atomic item ID -> semantic row
    """
    if dm.semantic_ids is None:
        raise ValueError("semantic_ids are required for semantic-ID candidate sampling.")

    # Collator/model semantic rows are ordered by sorted raw item ID.
    raw_items = sorted(dm.semantic_ids)

    item_semantic_ids = torch.tensor(
        [dm.semantic_ids[raw_item] for raw_item in raw_items],
        dtype=torch.long,
    )

    item_id_by_semantic_row = torch.tensor(
        [dm.item2id.get(raw_item, -1) for raw_item in raw_items],
        dtype=torch.long,
    )

    item_id_to_semantic_row = torch.full(
        (dm.item_vocab_size,),
        -1,
        dtype=torch.long,
    )
    for row, item_id in enumerate(item_id_by_semantic_row.tolist()):
        if item_id >= 0:
            item_id_to_semantic_row[item_id] = row

    return item_semantic_ids, item_id_by_semantic_row, item_id_to_semantic_row


def build_random_sampler(num_negative_samples: int = 8) -> RandomSemanticIDCandidateSampler:
    return RandomSemanticIDCandidateSampler(
        RandomSemanticCandidateConfig(
            num_negative_samples=num_negative_samples,
            include_positive=True,
            avoid_positive=True,
            pad_token_id=0,
            semantic_ids_are_fixed_length=True,
            max_resample_attempts=10,
        )
    )


def build_amazon_log_reward_model(
    *,
    item_id_by_semantic_row: torch.Tensor,
) -> AmazonLogRewardModel:
    return AmazonLogRewardModel(
        AmazonLogRewardConfig(
            # Candidate ID resolution.
            candidate_item_key="candidate_item_ids",
            pad_item_id=0,

            # Batch keys produced by the updated data pipeline.
            history_item_key="history_item_ids",
            history_rating_key="history_ratings",
            history_verified_key="history_verified_purchase",
            history_mask_key="history_mask",
            target_rating_key="target_ratings",
            target_verified_key="target_verified_purchase",

            # Reward engineering.
            # 1-3 stars -> near zero, 4 stars -> medium, 5 stars -> high.
            rating_mode="raw",

            # GFlowGR-style log-compressed reward scale.
            transform="log1p",
            log_scale=10.0,

            # Verified purchases receive a confidence boost.
            verified_bonus=0.25,
            unverified_multiplier=1.0,

            # If the same item appears multiple times in history, average its signal.
            history_aggregation="mean",

            # Truly unobserved random catalog items receive a very small reward.
            unobserved_reward=1e-3,
            reward_epsilon=1e-6,

            interaction_weight=1.0,

            require_target_rating=True,
            require_history_features=True,
            allow_missing_candidate_item_ids=False,
        ),
        item_id_by_semantic_row=item_id_by_semantic_row,
    )


def check_batch_has_reward_fields(batch: dict) -> None:
    required = [
        "history_item_ids",
        "history_ratings",
        "target_item_ids",
        "target_ratings",
        "target_token_ids",
        "target_token_mask",
    ]

    optional = [
        "history_verified_purchase",
        "target_verified_purchase",
        "history_mask",
        "history_semantic_row_ids",
    ]

    missing = [key for key in required if key not in batch]
    if missing:
        raise RuntimeError(
            "Batch is missing fields required by AmazonLogRewardModel: "
            f"{missing}. Make sure the updated data pipeline/collator is installed."
        )

    print("Required batch fields found.")
    for key in required + optional:
        if key in batch and torch.is_tensor(batch[key]):
            print(f"{key:30s} {tuple(batch[key].shape)} {batch[key].dtype}")
        elif key in batch:
            print(f"{key:30s} {type(batch[key])}")


def debug_one_batch(dataset_name: str = "beauty") -> None:
    dm = build_amazon_datamodule(dataset_name)
    item_semantic_ids, item_id_by_semantic_row, _ = build_semantic_tables(dm)

    train_loader = dm.train_dataloader()
    batch = next(iter(train_loader))
    check_batch_has_reward_fields(batch)

    sampler = build_random_sampler(num_negative_samples=8)
    reward_model = build_amazon_log_reward_model(
        item_id_by_semantic_row=item_id_by_semantic_row,
    )

    trajectories = sampler.sample(
        batch,
        item_semantic_ids=item_semantic_ids,
    )

    rewards = reward_model(batch, trajectories)

    print()
    print("Trajectory / reward diagnostics")
    print("candidate_token_ids:", tuple(trajectories.candidate_token_ids.shape))
    print("candidate_token_mask:", tuple(trajectories.candidate_token_mask.shape))
    print("is_positive:", tuple(trajectories.is_positive.shape))
    print("candidate_item_row_ids:", tuple(trajectories.candidate_item_row_ids.shape))
    print("rewards:", tuple(rewards.shape))
    print("reward min:", float(rewards.min()))
    print("reward mean:", float(rewards.mean()))
    print("reward max:", float(rewards.max()))

    # Optional: inspect first user's candidates.
    b = 15
    print()
    print("First example:")
    print("history item ids:", batch["history_item_ids"][b].tolist())
    print("history ratings:", batch["history_ratings"][b].tolist())
    if "history_verified_purchase" in batch:
        print("history verified:", batch["history_verified_purchase"][b].tolist())
    print("target item id:", int(batch["target_item_ids"][b]))
    print("target rating:", float(batch["target_ratings"][b]))
    if "target_verified_purchase" in batch:
        print("target verified:", bool(batch["target_verified_purchase"][b]))

    print("candidate rows:", trajectories.candidate_item_row_ids[b].tolist())
    print("is positive:", trajectories.is_positive[b].tolist())
    print("rewards:", rewards[b].tolist())
    


def build_tiger_model_for_dm(
    dm: AmazonReviewDataModule,
    *,
    item_semantic_ids: torch.Tensor,
) -> TigerModel:
    """
    Adjust this constructor to your repo's exact TigerModel signature.

    The important part is that the model must expose item_semantic_ids, because
    RandomSemanticIDCandidateSampler resolves its sampling table from
    model.item_semantic_ids when it is called through GFlowMethod.
    """
    if dm.sid_len is None:
        raise ValueError("dm.sid_len is required for TIGER.")

    model = TigerModel(
        codebook_size=256,
        sem_id_len=dm.sid_len,
        user_ids_count=2000,

        embedding_dim=128,
        num_heads=6,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=1024,
        d_kv=64,
        dropout=0.1,
        activation="relu",
        layer_norm_eps=1e-6,
        initializer_range=0.02,

        num_beams=100,
        num_return_sequences=20,

        item_semantic_ids=item_semantic_ids,
    )

    return model


def train_with_random_sampling_and_amazon_log_reward(
    dataset_name: str = "beauty",
    *,
    seed: int = 42,
) -> None:
    dm = build_amazon_datamodule(dataset_name, seed=seed)

    item_semantic_ids, item_id_by_semantic_row, _ = build_semantic_tables(dm)

    model = build_tiger_model_for_dm(
        dm,
        item_semantic_ids=item_semantic_ids,
    )

    reward_model = build_amazon_log_reward_model(
        item_id_by_semantic_row=item_id_by_semantic_row,
    )

    # Use the built-in random sampler path in GFlowMethod.
    # It samples from model.item_semantic_ids and passes candidate rows to the reward model.
    gflow_cfg = GFlowConfig(
        objective="tb",

        candidate_sampler="random",
        num_negative_samples=8,
        include_positive=True,
        avoid_positive=True,

        reward_epsilon=1e-6,

        sft_weight=1.0,
        gflow_weight=1.0,

        # Start simple. Use semantic_history later if you set logz_vocab_size.
        logz_context="global",
        logz_init=0.0,

        compute_train_metrics=True,
        compute_eval_metrics=True,
        compute_token_metrics=True,

        evaluate_generation=True,
        generation_topk=(5, 10, 20),
        generation_num_beams=100,
        generation_num_return_sequences=20,
        generation_constrained=True,

        metric_prefix="gflow",
    )

    method = GFlowMethod(
        cfg=gflow_cfg,
        reward_model=reward_model,
    )

    trainer_cfg = TrainerConfig(
        max_epochs=40,
        gradient_accumulation_steps=2,
        max_grad_norm=1.0,

        validate_every_epochs=2,
        log_every_steps=200,

        monitor="val/sft/ndcg@10",
        monitor_mode="max",
        early_stopping_patience=10,

        checkpoint_dir=f"../checkpoints/{dataset_name}_amazon_log_reward_random",
        save_best=True,
        save_last=True,
    )

    trainer = Trainer.from_optimizer_config(
        model=model,
        method=method,
        cfg=trainer_cfg,
        lr=3e-4,
        weight_decay=0.01,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

    trainer.fit(
        train_loader=dm.train_dataloader(),
        val_loader=dm.val_dataloader(),
    )


In [42]:
debug_one_batch("instruments")

Required batch fields found.
history_item_ids               (256, 20) torch.int64
history_ratings                (256, 20) torch.float32
target_item_ids                (256,) torch.int64
target_ratings                 (256,) torch.float32
target_token_ids               (256, 4) torch.int64
target_token_mask              (256, 4) torch.bool
history_verified_purchase      (256, 20) torch.bool
target_verified_purchase       (256,) torch.bool
history_mask                   (256, 20) torch.bool
history_semantic_row_ids       (256, 20) torch.int64

Trajectory / reward diagnostics
candidate_token_ids: (256, 9, 4)
candidate_token_mask: (256, 9, 4)
is_positive: (256, 9)
candidate_item_row_ids: (256, 9)
rewards: (256, 9)
reward min: 0.0010000000474974513
reward mean: 0.4452159106731415
reward max: 4.151040077209473

First example:
history item ids: [0, 0, 0, 0, 0, 0, 0, 2325, 9286, 6789, 2327, 1217, 865, 131, 6365, 1086, 1568, 5269, 47, 9287]
history ratings: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 